## Goal

The objective of this notebook is to identify and resolve data quality issues before feature preprocessing and model development. Cleaning decisions are documented to ensure reproducibility and maintain data integrity throughout the project.

### Task 1 
Question 1 (Business Perspective)

Imagine the company manager asks you:

*Why do we perform data cleaning in the first place? Can't the model work with the data as is?*

From a business perspective, what are 3 to 5 reasons you would give?

- Data cleaning improves data reliability before model training. Poor-quality data can lead to inaccurate predictions, unreliable business decisions, increased operational costs, and reduced customer retention performance.

Question 2 (Engineering Perspective)

Now, considering the EDA, what issues do you think exist in the dataset?

#### Data Quality Issues
    - TotalCharges is stored as string instead of numeric.
    - Hidden blank string values exist in TotalCharges.
    - customerID is not useful for prediction.
    - Numerical and categorical features require different preprocessing strategies.

Question 3

If you were to design a notebook for data cleaning, what would be the order of its sections?
```
Imports

↓

Load Dataset

↓

Dataset Overview

↓

Data Type Inspection

↓

Identify Data Quality Issues

↓

Handle Missing Values

↓

Convert Data Types

↓

Remove Irrelevant Features

↓

Save Clean Dataset

↓

Summary of Cleaning
```

In [18]:
# Imports 
import numpy as np
import pandas as pd

In [19]:
# Loading DataFrame
raw_df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
raw_df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [20]:
clean_df = raw_df.copy()

In [21]:
# Dataset Overview
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [22]:
# Data Type Inspection
print(clean_df.columns)

print(clean_df.dtypes)

print(clean_df.select_dtypes(include="object").columns)

print(clean_df.select_dtypes(include=np.number).columns)

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
Index(['customerID',

C:\Users\User\AppData\Local\Temp\ipykernel_11324\180363351.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(clean_df.select_dtypes(include="object").columns)


Most feature data types are appropriate. However, TotalCharges is stored as a string instead of a numeric type, and SeniorCitizen is represented as an integer although it behaves as a binary categorical feature.

Although SeniorCitizen represents a categorical concept, it is intentionally kept as a binary numeric feature because it is already machine-readable.

### Data Quality Investigation

- TotalCharges stored as string
- Blank strings detected
- customerID is an identifier
- SeniorCitizen is binary numeric

In [24]:
# Data Cleaning
clean_df['TotalCharges'] = pd.to_numeric(arg=clean_df['TotalCharges'], errors='coerce')
print(f'Before Dropping NaN Values: {clean_df.shape}')
clean_df.dropna(axis=0, inplace=True)
print(f'After Dropping NaN Values: {clean_df.shape}')
clean_df = clean_df.drop(columns=['customerID'])

Before Dropping NaN Values: (7032, 21)
After Dropping NaN Values: (7032, 21)


In [25]:
clean_df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [26]:
# Save Clean Dataset
clean_df.to_csv("../data/processed/customer_churn_clean.csv", index=False)

## Summary

Removed 11 records (0.16% of the dataset) due to missing values in TotalCharges

Data cleaning successfully resolved the primary data quality issues identified during EDA. The dataset is now ready for feature preprocessing in the next sprint.

## Sprint Retrospective

### Completed

- Investigated data quality issues
- Converted TotalCharges to numeric
- Handled missing values
- Removed non-predictive features
- Saved cleaned dataset

### Key Decisions

- Preserved the raw dataset
- Performed cleaning on a copied DataFrame
- Saved cleaned data separately

### Challenges

- Hidden blank strings in TotalCharges
- Data type inconsistency

### Next Sprint

Feature Encoding & Preprocessing